# Airfoil Self-Noise Regression Models

This notebook contains the supplied regression code separated into clearly labelled sections. It includes **Elastic Net, Polynomial Regression, KNN Regression, KNN hyperparameter tuning, SVR, and tuned SVR**.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")

: 

## 2. Load the Airfoil Self-Noise Dataset

In [ ]:
file_path = "../data/airfoil_self_noise.dat"

df = pd.read_csv(
    file_path,
    sep=r"\s+",
    header=None
)

df.head()

## 3. Prepare Training and Testing Data

The supplied model cells use `X_train`, `X_test`, `X_train_scaled`, `X_test_scaled`, `y_train`, `y_test`, and `final_features`. The original preprocessing cells were not included in the code you provided, so this section creates those variables using all five Airfoil features, an 80/20 split, and `StandardScaler`.

The UCI Airfoil Self-Noise dataset contains 1,503 observations, five input features, and one target: scaled sound pressure level. citeturn0search0

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_names = [
    "Frequency",
    "Angle_of_attack",
    "Chord_length",
    "Free_stream_velocity",
    "Suction_side_displacement_thickness"
]
target_name = "Sound_pressure_level"

df.columns = feature_names + [target_name]

X = df[feature_names]
y = df[target_name]

final_features = feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))
print("Number of features:", X_train.shape[1])

## 4. Import Regression Evaluation Metrics

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

## 5. Elastic Net Regression

In [ ]:
from sklearn.linear_model import ElasticNet

elastic_model = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    random_state=42
)

elastic_model.fit(
    X_train_scaled,
    y_train
)

y_train_pred_elastic = elastic_model.predict(
    X_train_scaled
)

y_test_pred_elastic = elastic_model.predict(
    X_test_scaled
)

In [ ]:
train_r2_elastic = r2_score(
    y_train,
    y_train_pred_elastic
)

test_r2_elastic = r2_score(
    y_test,
    y_test_pred_elastic
)

train_rmse_elastic = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred_elastic
    )
)

test_rmse_elastic = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred_elastic
    )
)

train_mae_elastic = mean_absolute_error(
    y_train,
    y_train_pred_elastic
)

test_mae_elastic = mean_absolute_error(
    y_test,
    y_test_pred_elastic
)

print("ElasticNet Regression")
print("----------------------------")
print("Train R²  :", train_r2_elastic)
print("Test R²   :", test_r2_elastic)
print("Train RMSE:", train_rmse_elastic)
print("Test RMSE :", test_rmse_elastic)
print("Train MAE :", train_mae_elastic)
print("Test MAE  :", test_mae_elastic)

In [ ]:
elastic_coefficients = pd.DataFrame({
    "Feature": final_features,
    "Coefficient": elastic_model.coef_
})

elastic_coefficients

## 6. Polynomial Regression

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

polynomial_model = Pipeline([
    ("polynomial_features", PolynomialFeatures(
        degree=2,
        include_bias=False
    )),
    ("linear_regression", LinearRegression())
])

polynomial_model.fit(
    X_train_scaled,
    y_train
)

y_train_pred_poly = polynomial_model.predict(
    X_train_scaled
)

y_test_pred_poly = polynomial_model.predict(
    X_test_scaled
)

In [ ]:
train_r2_poly = r2_score(
    y_train,
    y_train_pred_poly
)

test_r2_poly = r2_score(
    y_test,
    y_test_pred_poly
)

train_rmse_poly = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred_poly
    )
)

test_rmse_poly = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred_poly
    )
)

train_mae_poly = mean_absolute_error(
    y_train,
    y_train_pred_poly
)

test_mae_poly = mean_absolute_error(
    y_test,
    y_test_pred_poly
)

print("Polynomial Regression")
print("----------------------------")
print("Train R²  :", train_r2_poly)
print("Test R²   :", test_r2_poly)
print("Train RMSE:", train_rmse_poly)
print("Test RMSE :", test_rmse_poly)
print("Train MAE :", train_mae_poly)
print("Test MAE  :", test_mae_poly)

### 6.1 Polynomial Regression — Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_test_pred_poly,
    alpha=0.6
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.xlabel("Actual Sound Pressure Level (dB)")
plt.ylabel("Predicted Sound Pressure Level (dB)")
plt.title("Polynomial Regression: Actual vs Predicted")

plt.tight_layout()
plt.show()

### 6.2 Polynomial Regression — Residual Plot

In [ ]:
residuals_poly = y_test - y_test_pred_poly

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_pred_poly,
    residuals_poly,
    alpha=0.6
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Predicted Sound Pressure Level (dB)")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Polynomial Regression: Residual Plot")

plt.tight_layout()
plt.show()

## 7. KNN Regression

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_model = KNeighborsRegressor(
    n_neighbors=5
)

knn_model.fit(
    X_train_scaled,
    y_train
)

y_train_pred_knn = knn_model.predict(
    X_train_scaled
)

y_test_pred_knn = knn_model.predict(
    X_test_scaled
)

In [ ]:
train_r2_knn = r2_score(
    y_train,
    y_train_pred_knn
)

test_r2_knn = r2_score(
    y_test,
    y_test_pred_knn
)

train_rmse_knn = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred_knn
    )
)

test_rmse_knn = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred_knn
    )
)

train_mae_knn = mean_absolute_error(
    y_train,
    y_train_pred_knn
)

test_mae_knn = mean_absolute_error(
    y_test,
    y_test_pred_knn
)

print("KNN Regression")
print("----------------------------")
print("Train R²  :", train_r2_knn)
print("Test R²   :", test_r2_knn)
print("Train RMSE:", train_rmse_knn)
print("Test RMSE :", test_rmse_knn)
print("Train MAE :", train_mae_knn)
print("Test MAE  :", test_mae_knn)

### 7.1 KNN — Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_test_pred_knn,
    alpha=0.6
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.xlabel("Actual Sound Pressure Level (dB)")
plt.ylabel("Predicted Sound Pressure Level (dB)")
plt.title("KNN Regression: Actual vs Predicted")

plt.tight_layout()
plt.show()

### 7.2 KNN — Residual Plot

In [ ]:
residuals_knn = y_test - y_test_pred_knn

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_pred_knn,
    residuals_knn,
    alpha=0.6
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Predicted Sound Pressure Level (dB)")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("KNN Regression: Residual Plot")

plt.tight_layout()
plt.show()

## 8. KNN Hyperparameter Tuning — Manual Evaluation of k

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 20, 25, 30]

knn_results = []

for k in k_values:
    model = KNeighborsRegressor(
        n_neighbors=k
    )

    model.fit(
        X_train_scaled,
        y_train
    )

    predictions = model.predict(
        X_test_scaled
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    knn_results.append({
        "k": k,
        "R²": r2,
        "RMSE": rmse,
        "MAE": mae
    })

knn_tuning_results = pd.DataFrame(
    knn_results
)

knn_tuning_results

### 8.1 KNN Tuning — Test RMSE

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    knn_tuning_results["k"],
    knn_tuning_results["RMSE"],
    marker="o"
)

plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Test RMSE")
plt.title("KNN Hyperparameter Tuning")

plt.tight_layout()
plt.show()

### 8.2 KNN Tuning — Test R²

In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    knn_tuning_results["k"],
    knn_tuning_results["R²"],
    marker="o"
)

plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Test R²")
plt.title("KNN Hyperparameter Tuning")

plt.tight_layout()
plt.show()

## 9. KNN GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

k_values = [1, 3, 5, 7, 9, 11, 15, 20, 25, 30]

knn = KNeighborsRegressor()

param_grid = {
    "n_neighbors": k_values
}

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(
    X_train_scaled,
    y_train
)

print("Best k:", grid_search.best_params_["n_neighbors"])
print("Best Cross-Validation RMSE:",
      -grid_search.best_score_)

In [ ]:
best_knn = grid_search.best_estimator_

y_train_pred_knn_tuned = best_knn.predict(
    X_train_scaled
)

y_test_pred_knn_tuned = best_knn.predict(
    X_test_scaled
)

train_r2_knn_tuned = r2_score(
    y_train,
    y_train_pred_knn_tuned
)

test_r2_knn_tuned = r2_score(
    y_test,
    y_test_pred_knn_tuned
)

train_rmse_knn_tuned = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred_knn_tuned
    )
)

test_rmse_knn_tuned = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred_knn_tuned
    )
)

train_mae_knn_tuned = mean_absolute_error(
    y_train,
    y_train_pred_knn_tuned
)

test_mae_knn_tuned = mean_absolute_error(
    y_test,
    y_test_pred_knn_tuned
)

print("Tuned KNN Regression")
print("----------------------------")
print("Best k     :", grid_search.best_params_["n_neighbors"])
print("Train R²   :", train_r2_knn_tuned)
print("Test R²    :", test_r2_knn_tuned)
print("Train RMSE :", train_rmse_knn_tuned)
print("Test RMSE  :", test_rmse_knn_tuned)
print("Train MAE  :", train_mae_knn_tuned)
print("Test MAE   :", test_mae_knn_tuned)

## 10. Support Vector Regression (SVR)

In [ ]:
from sklearn.svm import SVR

svr_model = SVR(
    kernel="rbf",
    C=100,
    epsilon=0.1,
    gamma="scale"
)

svr_model.fit(
    X_train_scaled,
    y_train
)

y_train_pred_svr = svr_model.predict(
    X_train_scaled
)

y_test_pred_svr = svr_model.predict(
    X_test_scaled
)

In [ ]:
train_r2_svr = r2_score(
    y_train,
    y_train_pred_svr
)

test_r2_svr = r2_score(
    y_test,
    y_test_pred_svr
)

train_rmse_svr = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred_svr
    )
)

test_rmse_svr = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred_svr
    )
)

train_mae_svr = mean_absolute_error(
    y_train,
    y_train_pred_svr
)

test_mae_svr = mean_absolute_error(
    y_test,
    y_test_pred_svr
)

print("SVR")
print("----------------------------")
print("Train R²  :", train_r2_svr)
print("Test R²   :", test_r2_svr)
print("Train RMSE:", train_rmse_svr)
print("Test RMSE :", test_rmse_svr)
print("Train MAE :", train_mae_svr)
print("Test MAE  :", test_mae_svr)

### 10.1 SVR — Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_test_pred_svr,
    alpha=0.6
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.xlabel("Actual Sound Pressure Level (dB)")
plt.ylabel("Predicted Sound Pressure Level (dB)")
plt.title("SVR: Actual vs Predicted")

plt.tight_layout()
plt.show()

### 10.2 SVR — Residual Plot

In [ ]:
residuals_svr = y_test - y_test_pred_svr

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_pred_svr,
    residuals_svr,
    alpha=0.6
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Predicted Sound Pressure Level (dB)")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("SVR: Residual Plot")

plt.tight_layout()
plt.show()

## 11. SVR GridSearchCV Hyperparameter Tuning

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf"))
])

param_grid_svr = {
    "svr__C": [1, 10, 100, 300],
    "svr__epsilon": [0.01, 0.1, 0.5],
    "svr__gamma": ["scale", 0.01, 0.1]
}

svr_grid = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=param_grid_svr,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

svr_grid.fit(
    X_train,
    y_train
)

print("Best Parameters:")
print(svr_grid.best_params_)

print("\nBest Cross-Validation RMSE:")
print(-svr_grid.best_score_)

In [ ]:
best_svr = svr_grid.best_estimator_

y_train_pred_svr_tuned = best_svr.predict(
    X_train
)

y_test_pred_svr_tuned = best_svr.predict(
    X_test
)

print("Tuned SVR")
print("----------------------------")

print(
    "Best Parameters:",
    svr_grid.best_params_
)

print(
    "Train R²:",
    r2_score(
        y_train,
        y_train_pred_svr_tuned
    )
)

print(
    "Test R²:",
    r2_score(
        y_test,
        y_test_pred_svr_tuned
    )
)

print(
    "Train RMSE:",
    np.sqrt(
        mean_squared_error(
            y_train,
            y_train_pred_svr_tuned
        )
    )
)

print(
    "Test RMSE:",
    np.sqrt(
        mean_squared_error(
            y_test,
            y_test_pred_svr_tuned
        )
    )
)

print(
    "Train MAE:",
    mean_absolute_error(
        y_train,
        y_train_pred_svr_tuned
    )
)

print(
    "Test MAE:",
    mean_absolute_error(
        y_test,
        y_test_pred_svr_tuned
    )
)

### 11.1 Tuned SVR — Actual vs Predicted

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_test_pred_svr_tuned,
    alpha=0.6
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.xlabel("Actual Sound Pressure Level (dB)")
plt.ylabel("Predicted Sound Pressure Level (dB)")
plt.title("Tuned SVR: Actual vs Predicted")

plt.tight_layout()
plt.show()

### 11.2 Tuned SVR — Residual Plot

In [ ]:
residuals_svr_tuned = (
    y_test - y_test_pred_svr_tuned
)

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test_pred_svr_tuned,
    residuals_svr_tuned,
    alpha=0.6
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Predicted Sound Pressure Level (dB)")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Tuned SVR: Residual Plot")

plt.tight_layout()
plt.show()

## 12. Model Results Summary

This final cell collects the test metrics from all six model variants after the preceding cells have been executed.

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Elastic Net",
        "Polynomial Regression",
        "KNN (k=5)",
        "Tuned KNN",
        "SVR",
        "Tuned SVR"
    ],
    "Test R²": [
        test_r2_elastic,
        test_r2_poly,
        test_r2_knn,
        test_r2_knn_tuned,
        test_r2_svr,
        r2_score(y_test, y_test_pred_svr_tuned)
    ],
    "Test RMSE": [
        test_rmse_elastic,
        test_rmse_poly,
        test_rmse_knn,
        test_rmse_knn_tuned,
        test_rmse_svr,
        np.sqrt(mean_squared_error(y_test, y_test_pred_svr_tuned))
    ],
    "Test MAE": [
        test_mae_elastic,
        test_mae_poly,
        test_mae_knn,
        test_mae_knn_tuned,
        test_mae_svr,
        mean_absolute_error(y_test, y_test_pred_svr_tuned)
    ]
})

results